In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")

In [0]:
env = dbutils.widgets.get("environment")
print(env)
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawinvoice"
# print(brznTablName)
silverTablName = f"saleslake_{env}.silver_{env}.cleaninvoice"
# print(brznTablName)

In [0]:
# from pyspark.sql import functions as F

# # Read bronze table
# bronze_df = spark.table(bronzeTablName)

# # Apply transformations
# silver_df = (
#     bronze_df
#     .select(
#         F.trim("invoice_id").alias("invoice_id"),
#         F.trim("invoice_number").alias("invoice_number"),
#         F.col("customer_id").alias("customer_id"),
#         F.to_date(F.trim("invoice_date"), 'yyyy-MM-dd').alias("invoice_date"),
#         F.to_date(F.trim("due_date"), 'yyyy-MM-dd').alias("due_date"),
#         F.col("subtotal_amount").cast("decimal(18,2)").alias("subtotal_amount"),
#         F.upper(F.when(F.trim("discount_code") == "", None)
#                 .otherwise(F.trim("discount_code"))).alias("discount_code"),
#         F.col("discount_amount").cast("decimal(18,2)").alias("discount_amount"),
#         F.col("tax_amount").cast("decimal(18,2)").alias("tax_amount"),
#         F.col("total_amount").cast("decimal(18,2)").alias("total_amount"),
#         F.upper(F.trim("payment_status")).alias("payment_status"),
#         F.upper(F.when(F.trim("payment_method") == "", None)
#                 .otherwise(F.trim("payment_method"))).alias("payment_method"),
#         F.when(F.trim("payment_date") == "", None)
#          .otherwise(F.to_date(F.trim("payment_date"), 'yyyy-MM-dd')).alias("payment_date"),
#         F.upper(F.trim("currency")).alias("currency"),
#         F.upper(F.trim("region")).alias("region"),
#         F.upper(F.trim("store_id")).alias("store_id"),
#         F.upper(F.trim("channel")).alias("channel"),
#         F.lower(F.trim("created_by")).alias("created_by"),
#         F.current_timestamp().alias("ingest_ts")
#     )
#     .distinct()
# )

# # Filter based on ingest_ts (incremental load)
# max_ingest_ts = spark.table(silverTablName).agg(
#     F.coalesce(F.max("ingest_ts"), F.to_timestamp(F.lit("1990-01-01"), "yyyy-MM-dd"))
# ).collect()[0][0]

# silver_df_filtered = silver_df.filter(F.col("ingest_ts") > max_ingest_ts)

# # Write into silver table
# (silver_df_filtered
#     .orderBy(F.trim("invoice_id"))
#     .write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(silverTablName)
# )


In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    TRIM(invoice_id)                                              AS invoice_id,
    TRIM(invoice_number)                                          AS invoice_number,
    CAST(TRIM(customer_id) AS BIGINT)                             AS customer_id,
    TO_DATE(TRIM(invoice_date),  'yyyy-MM-dd')                    AS invoice_date,
    TO_DATE(TRIM(due_date),      'yyyy-MM-dd')                    AS due_date,
    CAST(TRIM(subtotal_amount) AS DECIMAL(18,2))                  AS subtotal_amount,
    UPPER(NULLIF(TRIM(discount_code), ''))                        AS discount_code,
    CAST(TRIM(discount_amount) AS DECIMAL(18,2))                  AS discount_amount,
    CAST(TRIM(tax_amount)      AS DECIMAL(18,2))                  AS tax_amount,
    CAST(TRIM(total_amount)    AS DECIMAL(18,2))                  AS total_amount,
    UPPER(TRIM(payment_status))                                   AS payment_status,
    UPPER(NULLIF(TRIM(payment_method), ''))                       AS payment_method,
    CASE
        WHEN NULLIF(TRIM(payment_date), '') IS NULL THEN NULL
        ELSE TO_DATE(TRIM(payment_date), 'yyyy-MM-dd')
    END                                                           AS payment_date,
    UPPER(TRIM(currency))                                         AS currency,
    UPPER(TRIM(region))                                           AS region,
    UPPER(TRIM(store_id))                                         AS store_id,
    UPPER(TRIM(channel))                                          AS channel,
    LOWER(TRIM(created_by))                                       AS created_by,
    CURRENT_TIMESTAMP()                                           AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd'))
    FROM {silverTablName}
)
ORDER BY TRIM(invoice_id)
""")

In [0]:
%sql
SELECT * FROM saleslake_prod.silver_prod.cleaninvoice;



In [0]:
%sql
SELECT * FROM saleslake_dev.silver_dev.cleanInvoice;

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.silver_dev.cleanInvoice;
select count(*) from saleslake_prod.bronze_prod.rawInvoice ;

In [0]:
%sql
DESCRIBE HISTORY saleslake_dev.silver_dev.cleanInvoice;